In [43]:
import conllu
import pandas as pd

In [87]:
sentences = []
with open("train_nlprepl-ud.conllu", "r", encoding="utf-8") as f:
    for sentence in conllu.parse_incr(f):
        sentences.append(sentence)

In [42]:
print(len(sentences))
print(sentences[0].metadata['text'])

69360
Zatrzasnął drzwi od mieszkania, dwa razy przekręcił klucz, nacisnął klamkę, by sprawdzić, czy dobrze zamknięte, zbiegł po schodach, minął furtkę, także ją zamknął, i znalazł się na wąskiej uliczce między ogródkami, gdzie drzemały w majowym słońcu trójkątne ciemnozielone świerki, jakich nie było w pobliżu jego domu.


In [58]:
sentence = sentences[4]

for t in sentence:
    print(t.items())


dict_items([('id', 1), ('form', 'Swoją'), ('lemma', 'swój'), ('upos', 'DET'), ('xpos', 'adj:sg:acc:f:pos'), ('feats', {'Case': 'Acc', 'Gender': 'Fem', 'Number': 'Sing', 'Poss': 'Yes', 'PronType': 'Prs', 'Reflex': 'Yes'}), ('head', 2), ('deprel', 'advmod:emph'), ('deps', [('advmod:emph', 2)]), ('misc', None)])
dict_items([('id', 2), ('form', 'drogę'), ('lemma', 'droga'), ('upos', 'NOUN'), ('xpos', 'subst:sg:acc:f'), ('feats', {'Case': 'Acc', 'Gender': 'Fem', 'Number': 'Sing'}), ('head', 11), ('deprel', 'obj'), ('deps', [('obj', 11)]), ('misc', None)])
dict_items([('id', 3), ('form', 'do'), ('lemma', 'do'), ('upos', 'ADP'), ('xpos', 'prep:gen'), ('feats', {'AdpType': 'Prep'}), ('head', 5), ('deprel', 'case'), ('deps', [('case', 5)]), ('misc', {'Case': 'Gen'})])
dict_items([('id', 4), ('form', 'tego'), ('lemma', 'ten'), ('upos', 'DET'), ('xpos', 'adj:sg:gen:n:pos'), ('feats', {'Case': 'Gen', 'Gender': 'Neut', 'Number': 'Sing', 'PronType': 'Dem'}), ('head', 5), ('deprel', 'det'), ('deps', 

In [30]:
tree = sentence.to_tree()
tree.print_tree()

(deprel:root) form:zaczął lemma:zacząć upos:VERB [11]
    (deprel:obj) form:drogę lemma:droga upos:NOUN [2]
        (deprel:advmod:emph) form:Swoją lemma:swój upos:DET [1]
        (deprel:nmod) form:miasta lemma:miasto upos:NOUN [5]
            (deprel:case) form:do lemma:do upos:ADP [3]
            (deprel:det) form:tego lemma:ten upos:DET [4]
    (deprel:nsubj) form:autor lemma:autor upos:NOUN [6]
        (deprel:nmod) form:dni lemma:dzień upos:NOUN [9]
            (deprel:punct) form:" lemma:" upos:PUNCT [7]
            (deprel:amod) form:Krótkich lemma:krótki upos:ADJ [8]
            (deprel:punct) form:" lemma:" upos:PUNCT [10]
    (deprel:obl) form:daleka lemma:daleki upos:ADJ [14]
        (deprel:case) form:z lemma:z upos:ADP [12]
        (deprel:advmod) form:bardzo lemma:bardzo upos:ADV [13]
    (deprel:punct) form:. lemma:. upos:PUNCT [15]


### Conllu zawiera potrzebne tagi takie jak 'czas', 'forma', rodzaj' itd, ale nie dostarcza słownika jak skonwertować to do innych form/czasów. Do tego i tak bedzie potrzeba jakiegoś dodatkowego słownika (np. jak PoliMorf).

In [47]:
dict_df = pd.read_csv("dictionary.csv")
m_by_f = {row["praet:sg:f:perf"]: row["praet:sg:m1.m2.m3:perf"] for _, row in dict_df.iterrows()}
f_by_b = {row["praet:sg:m1.m2.m3:perf"]: row["praet:sg:f:perf"] for _, row in dict_df.iterrows()}

Predicate - token that:
* is root, or
* is the HEAD of a subject token (can be checked with 'deprel' dependency relation label)

In [98]:
from copy import deepcopy
from tqdm import tqdm

def sentence_text(sent):
    """Conlu library doesn't have 'generate raw text' function"""
    out = []
    for tok in sent:
        if not isinstance(tok["id"], int):
            continue

        out.append(tok["form"])

        misc = tok.get("misc")
        if not misc or misc.get("SpaceAfter") != "No":
            out.append(" ")

    return "".join(out).rstrip()

pairs = []

for sentence in tqdm(sentences):
    sentence = deepcopy(sentence)  # to not break original objects
    root = sentence.to_tree()

    upos = root.token['upos'] # Universal Part-of-Speech
    if upos != 'VERB':
        # For now lets take only root predicates
        continue

    xpos = root.token['xpos'] # language-specific part-of-speech tag

    correct_text = sentence.metadata['text']
    if xpos == 'praet:sg:f:perf':
        form = root.token['form']
        was_upper = form[0].isupper()
        form = form.lower()

        if form not in m_by_f:
            continue

        m_form = m_by_f[form]
        if was_upper:
            m_form = m_form[:1].upper() + m_form[1:]

        root.token['form'] = m_form
        root.token['xpos'] = "praet:sg:m1.m2.m3:perf"

        incorrect_text = sentence_text(sentence)
        pairs.append((correct_text, incorrect_text))
    else:
        # Prepare other scenarios here
        continue

df = pd.DataFrame(pairs, columns=["correct", "incorrect"])
df


100%|██████████| 69360/69360 [00:11<00:00, 5958.52it/s]


,correct,incorrect
0,"Otarła je lewą, umęczoną ręką.","Otarł je lewą, umęczoną ręką."
1,Prawie wesoła - dokończyła przy pomocy krokody...,Prawie wesoła - dokończył przy pomocy krokodyl...
2,"Strzepnęła ją, złożyła, odniosła tamże, gdzie ...","Strzepnął ją, złożyła, odniosła tamże, gdzie p..."
3,"Telefon zadzwonił, upudrowała się gorączkowo, ...","Telefon zadzwonił, upudrował się gorączkowo, p..."
4,"Ledwie zdjęła słuchawkę, Sabina przycwałowała ...","Ledwie zdjęła słuchawkę, Sabina przycwałował s..."
...,...,...
2695,A wczoraj poszłam do kina (z rodzicami i siost...,A wczoraj poszedł m do kina (z rodzicami i sio...
2696,Ostatnio zaczęłam oglądać filmy które nie wyma...,Ostatnio zaczął m oglądać filmy które nie wyma...
2697,"Niedawno skończyłam ""Dziecko Noego"" - Eric Emm...","Niedawno skończył m ""Dziecko Noego"" - Eric Emm..."
2698,"No i naturalnie przeczytałam ""Harry Potter i K...","No i naturalnie przeczytał m ""Harry Potter i K..."


## Problem - czasami zdanie przypadkowo ciągle jest poprawne, bo podmiot np. jest domyslny. Mając jednak tagi wszystkie takie rzeczy można sprawdzić.

In [99]:
df.to_csv("pairs.csv", index=False)